# VeReMi-Graph v1.0 — Dataset Characterization and Publication Statistics

This notebook generates the **publication-ready characterization outputs** for the proposed Data in Brief article:

**VeReMi-Graph: A Temporal Attributed Graph Dataset for Vehicular Misbehavior Detection**

### Input
Public release root:

`RELEASE_ROOT`

### Outputs
`RELEASE_ROOT / "publication_stats"`

The notebook produces:

- authoritative dataset summary;
- graph-size statistics;
- node and edge distributions;
- class distribution;
- split comparison;
- graph structural statistics;
- feature-family summary;
- storage summary;
- original VeReMi vs VeReMi-Graph comparison table;
- publication-ready CSV tables;
- publication-ready PNG figures.

No dataset files are modified.


In [ ]:
from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

from src.config import RELEASE_ROOT
ROOT = RELEASE_ROOT
DATA_ROOT = ROOT / "data"
NODE_DIR = DATA_ROOT / "nodes"
EDGE_DIR = DATA_ROOT / "edges"
SNAP_DIR = DATA_ROOT / "snapshots"
META_DIR = ROOT / "metadata"
SCHEMA_DIR = ROOT / "schemas"
PYG_DIR = ROOT / "pyg"
OUT = ROOT / "publication_stats"

TABLE_DIR = OUT / "tables"
FIG_DIR = OUT / "figures"

for p in [OUT, TABLE_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Publication statistics output:", OUT)


## 1. Load authoritative release metadata


In [ ]:
with open(META_DIR / "dataset_statistics.json", "r", encoding="utf-8") as f:
    dataset_stats = json.load(f)

archive_manifest = pd.read_csv(META_DIR / "archive_manifest.csv")
class_distribution = pd.read_csv(META_DIR / "class_distribution.csv")
split_distribution = pd.read_csv(META_DIR / "benchmark_split_class_distribution.csv")
feature_dictionary = pd.read_csv(META_DIR / "feature_dictionary.csv")

print(json.dumps(dataset_stats, indent=2))
display(archive_manifest.head())


## 2. Authoritative dataset summary table


In [ ]:
summary_table = pd.DataFrame([
    ["Dataset name", dataset_stats["dataset_name"]],
    ["Version", dataset_stats["dataset_version"]],
    ["Source dataset", dataset_stats["source_dataset"]],
    ["Source archives", dataset_stats["source_archive_count"]],
    ["Node records", dataset_stats["total_node_rows"]],
    ["Directed communication edges", dataset_stats["total_edge_rows"]],
    ["Temporal graph snapshots", dataset_stats["total_graph_snapshot_rows"]],
    ["Snapshot interval (s)", dataset_stats["snapshot_interval_seconds"]],
    ["Train archives", dataset_stats["train_archives"]],
    ["Validation archives", dataset_stats["validation_archives"]],
    ["Test archives", dataset_stats["test_archives"]],
    ["Primary prediction unit", dataset_stats["primary_prediction_unit"]],
    ["Multiclass target", dataset_stats["multiclass_target"]],
    ["Class values", ", ".join(map(str, dataset_stats["multiclass_values"]))],
], columns=["Property", "Value"])

summary_table.to_csv(TABLE_DIR / "Table_1_dataset_summary.csv", index=False)
display(summary_table)


## 3. Graph-size statistics across 22,500 snapshots


In [ ]:
pyg_manifest = pd.read_csv(PYG_DIR / "pyg_snapshot_manifest.csv")

graph_stats = pd.DataFrame({
    "Metric": [
        "Number of graph snapshots",
        "Mean nodes per graph",
        "Median nodes per graph",
        "Minimum nodes per graph",
        "Maximum nodes per graph",
        "Mean edges per graph",
        "Median edges per graph",
        "Minimum edges per graph",
        "Maximum edges per graph",
        "Mean edge-to-node ratio"
    ],
    "Value": [
        len(pyg_manifest),
        pyg_manifest["node_count"].mean(),
        pyg_manifest["node_count"].median(),
        pyg_manifest["node_count"].min(),
        pyg_manifest["node_count"].max(),
        pyg_manifest["edge_count"].mean(),
        pyg_manifest["edge_count"].median(),
        pyg_manifest["edge_count"].min(),
        pyg_manifest["edge_count"].max(),
        (pyg_manifest["edge_count"] / pyg_manifest["node_count"]).mean()
    ]
})

graph_stats.to_csv(TABLE_DIR / "Table_2_graph_size_statistics.csv", index=False)
display(graph_stats)


## 4. Distribution of nodes per temporal graph


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(pyg_manifest["node_count"], bins=30)
ax.set_xlabel("Nodes per graph snapshot")
ax.set_ylabel("Number of graph snapshots")
ax.set_title("Distribution of nodes per temporal graph")
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure_1_nodes_per_graph_distribution.png", dpi=300)
plt.show()


## 5. Distribution of communication edges per temporal graph


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(pyg_manifest["edge_count"], bins=30)
ax.set_xlabel("Directed communication edges per graph snapshot")
ax.set_ylabel("Number of graph snapshots")
ax.set_title("Distribution of communication edges per temporal graph")
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure_2_edges_per_graph_distribution.png", dpi=300)
plt.show()


## 6. Multiclass attack distribution


In [ ]:
label_name_map = {
    0: "Benign",
    1: "Attack 1",
    2: "Attack 2",
    4: "Attack 4",
    8: "Attack 8",
    16: "Attack 16"
}

class_pub = class_distribution.copy()
class_pub["Class"] = class_pub["attackerType"].map(label_name_map)
class_pub["Percentage"] = class_pub["fraction"] * 100

class_pub[["Class","edge_count","Percentage"]].to_csv(
    TABLE_DIR / "Table_3_class_distribution.csv",
    index=False
)

display(class_pub[["Class","edge_count","Percentage"]])

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(class_pub["Class"], class_pub["Percentage"])
ax.set_ylabel("Percentage of communication edges")
ax.set_xlabel("Class")
ax.set_title("VeReMi-Graph class distribution")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure_3_class_distribution.png", dpi=300)
plt.show()


## 7. Benchmark split class-distribution comparison


In [ ]:
split_pub = split_distribution.copy()
split_pub["Class"] = split_pub["attackerType"].map(label_name_map)
split_pub["Percentage"] = split_pub["fraction"] * 100

split_pivot = split_pub.pivot_table(
    index="Class",
    columns="split",
    values="Percentage"
).reset_index()

split_pivot.to_csv(
    TABLE_DIR / "Table_4_split_class_distribution.csv",
    index=False
)

display(split_pivot)

plot_df = split_pivot.set_index("Class")
fig, ax = plt.subplots(figsize=(9,5))
plot_df.plot(kind="bar", ax=ax)
ax.set_ylabel("Percentage of communication edges")
ax.set_xlabel("Class")
ax.set_title("Class distribution across archive-disjoint benchmark splits")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure_4_split_class_distribution.png", dpi=300)
plt.show()


## 8. Archive-level graph and storage characterization


In [ ]:
archive_stats = pd.DataFrame({
    "Metric": [
        "Mean node rows per archive",
        "Median node rows per archive",
        "Min node rows per archive",
        "Max node rows per archive",
        "Mean edge rows per archive",
        "Median edge rows per archive",
        "Min edge rows per archive",
        "Max edge rows per archive",
        "Mean unique vehicles per archive",
        "Median unique vehicles per archive",
        "Min unique vehicles per archive",
        "Max unique vehicles per archive",
    ],
    "Value": [
        archive_manifest["node_rows"].mean(),
        archive_manifest["node_rows"].median(),
        archive_manifest["node_rows"].min(),
        archive_manifest["node_rows"].max(),
        archive_manifest["edge_rows"].mean(),
        archive_manifest["edge_rows"].median(),
        archive_manifest["edge_rows"].min(),
        archive_manifest["edge_rows"].max(),
        archive_manifest["unique_vehicles"].mean(),
        archive_manifest["unique_vehicles"].median(),
        archive_manifest["unique_vehicles"].min(),
        archive_manifest["unique_vehicles"].max(),
    ]
})

archive_stats.to_csv(
    TABLE_DIR / "Table_5_archive_level_statistics.csv",
    index=False
)

display(archive_stats)


## 9. Graph structural statistics from snapshot layer


In [ ]:
snapshot_files = sorted(SNAP_DIR.glob("*.parquet"))

structural_cols = [
    "node_count",
    "isolated_node_count",
    "message_edge_count",
    "unique_directed_edge_count",
    "density",
    "reciprocity",
    "average_degree",
    "average_clustering",
    "connected_components",
    "largest_component_fraction",
    "mean_RSSI",
    "median_communication_delay",
]

parts = []

for i,p in enumerate(snapshot_files,1):
    df = pd.read_parquet(p)
    available = [c for c in structural_cols if c in df.columns]
    parts.append(df[available])

    if i == 1 or i % 50 == 0:
        print(f"Loaded snapshot statistics {i}/{len(snapshot_files)}")

snapshot_all = pd.concat(parts, ignore_index=True)

rows = []
for col in snapshot_all.columns:
    x = pd.to_numeric(snapshot_all[col], errors="coerce")
    rows.append({
        "Feature": col,
        "Mean": x.mean(),
        "Median": x.median(),
        "Std": x.std(),
        "Min": x.min(),
        "Max": x.max(),
        "Missing": int(x.isna().sum())
    })

structural_summary = pd.DataFrame(rows)
structural_summary.to_csv(
    TABLE_DIR / "Table_6_graph_structural_statistics.csv",
    index=False
)
display(structural_summary)


## 10. Feature-family summary


In [ ]:
role_counts = (
    feature_dictionary
    .groupby(["table","role"])
    .size()
    .reset_index(name="feature_count")
)

role_counts.to_csv(
    TABLE_DIR / "Table_7_feature_role_summary.csv",
    index=False
)

display(role_counts)

fig, ax = plt.subplots(figsize=(9,5))
plot = role_counts.pivot_table(
    index="role",
    columns="table",
    values="feature_count",
    fill_value=0
)
plot.plot(kind="bar", ax=ax)
ax.set_ylabel("Number of fields")
ax.set_xlabel("Feature role")
ax.set_title("VeReMi-Graph feature-role distribution")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure_5_feature_role_distribution.png", dpi=300)
plt.show()


## 11. Public storage footprint


In [ ]:
physical_summary = pd.read_csv(META_DIR / "public_data_layer_summary.csv")

pyg_size_gb = (
    pyg_manifest["file_size_bytes"].sum()
    / (1024**3)
)

storage_table = physical_summary.copy()
storage_table["format"] = "Parquet"

storage_table = storage_table.rename(
    columns={
        "layer":"Layer",
        "file_count":"Files",
        "row_count":"Rows",
        "size_gb":"Size_GiB"
    }
)

storage_table = storage_table[
    ["Layer","format","Files","Rows","Size_GiB"]
]

storage_table = pd.concat([
    storage_table,
    pd.DataFrame([{
        "Layer":"PyG snapshots",
        "format":"PyTorch Geometric .pt",
        "Files":len(pyg_manifest),
        "Rows":len(pyg_manifest),
        "Size_GiB":pyg_size_gb
    }])
], ignore_index=True)

storage_table.to_csv(
    TABLE_DIR / "Table_8_storage_footprint.csv",
    index=False
)

display(storage_table)


## 12. Original VeReMi vs VeReMi-Graph comparison


In [ ]:
comparison = pd.DataFrame([
    ["Primary representation", "Vehicle JSON/simulation logs", "Temporal attributed graphs"],
    ["Graph nodes", "Not explicitly provided", "Vehicles"],
    ["Directed sender→receiver edges", "Not explicitly reconstructed as graph edges", "39,136,291 communication edges"],
    ["Temporal graph snapshots", "Not provided as graph objects", "22,500 one-second graph snapshots"],
    ["Node feature layer", "Raw state records", "2,016,487 graph-node records"],
    ["Edge feature layer", "Raw received-message fields", "23 frozen model-safe edge features plus metadata/validation fields"],
    ["Node model-safe inputs", "Not defined", "26 frozen primary node features"],
    ["Graph context", "Not defined as graph context", "13 lagged graph-context features"],
    ["PyTorch Geometric representation", "Not provided", "22,500 ready-to-load .pt graph objects"],
    ["Benchmark split", "Not provided as archive-disjoint graph benchmark", "157/34/34 archive-disjoint train/validation/test split"],
    ["Class balancing", "Original source distribution", "Natural message distribution preserved; no synthetic balancing"],
    ["Leakage-control schema", "Not supplied", "Explicit target/identifier/provenance/leakage roles"],
    ["Checksums", "Source-dependent", "Release-level SHA-256 checksums"],
], columns=["Aspect","Original VeReMi","VeReMi-Graph v1.0"])

comparison.to_csv(
    TABLE_DIR / "Table_9_original_veremi_vs_veremi_graph.csv",
    index=False
)

display(comparison)


## 13. Binary benign vs malicious distribution


In [ ]:
benign = int(class_distribution.loc[class_distribution["attackerType"] == 0, "edge_count"].iloc[0])
total = int(class_distribution["edge_count"].sum())
malicious = total - benign

binary_table = pd.DataFrame({
    "Class":["Benign","Malicious"],
    "Edge_count":[benign, malicious],
    "Percentage":[100*benign/total, 100*malicious/total]
})

binary_table.to_csv(
    TABLE_DIR / "Table_10_binary_class_distribution.csv",
    index=False
)
display(binary_table)

fig, ax = plt.subplots(figsize=(6,5))
ax.bar(binary_table["Class"], binary_table["Percentage"])
ax.set_ylabel("Percentage of communication edges")
ax.set_title("Binary class distribution")
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure_6_binary_class_distribution.png", dpi=300)
plt.show()


## 14. Split graph counts


In [ ]:
split_graphs = (
    pyg_manifest
    .groupby("split")
    .agg(
        graph_snapshots=("snapshot_id","count"),
        node_instances=("node_count","sum"),
        edge_instances=("edge_count","sum")
    )
    .reset_index()
)

split_graphs.to_csv(
    TABLE_DIR / "Table_11_split_graph_counts.csv",
    index=False
)
display(split_graphs)


## 15. Generate a publication-statistics JSON summary


In [ ]:
publication_summary = {
    "dataset_name": "VeReMi-Graph",
    "version": "1.0",
    "source_archives": int(dataset_stats["source_archive_count"]),
    "node_records": int(dataset_stats["total_node_rows"]),
    "communication_edges": int(dataset_stats["total_edge_rows"]),
    "graph_snapshots": int(dataset_stats["total_graph_snapshot_rows"]),
    "snapshot_interval_seconds": float(dataset_stats["snapshot_interval_seconds"]),
    "node_feature_dimension": int(pyg_manifest["node_feature_count"].iloc[0]),
    "edge_feature_dimension": int(pyg_manifest["edge_feature_count"].iloc[0]),
    "graph_context_dimension": int(pyg_manifest["graph_context_feature_count"].iloc[0]),
    "mean_nodes_per_graph": float(pyg_manifest["node_count"].mean()),
    "median_nodes_per_graph": float(pyg_manifest["node_count"].median()),
    "min_nodes_per_graph": int(pyg_manifest["node_count"].min()),
    "max_nodes_per_graph": int(pyg_manifest["node_count"].max()),
    "mean_edges_per_graph": float(pyg_manifest["edge_count"].mean()),
    "median_edges_per_graph": float(pyg_manifest["edge_count"].median()),
    "min_edges_per_graph": int(pyg_manifest["edge_count"].min()),
    "max_edges_per_graph": int(pyg_manifest["edge_count"].max()),
    "train_graphs": int((pyg_manifest["split"]=="train").sum()),
    "validation_graphs": int((pyg_manifest["split"]=="validation").sum()),
    "test_graphs": int((pyg_manifest["split"]=="test").sum()),
    "pyg_size_gib": float(pyg_size_gb),
    "class_distribution": {
        str(int(r["attackerType"])): {
            "count": int(r["edge_count"]),
            "fraction": float(r["fraction"])
        }
        for _,r in class_distribution.iterrows()
    }
}

with open(
    OUT / "publication_statistics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(publication_summary, f, indent=2)

print(json.dumps(publication_summary, indent=2))


## 16. Final publication-statistics validation


In [ ]:
checks = [
    ("225 source archives", publication_summary["source_archives"] == 225),
    ("2,016,487 node records", publication_summary["node_records"] == 2_016_487),
    ("39,136,291 edge records", publication_summary["communication_edges"] == 39_136_291),
    ("22,500 graph snapshots", publication_summary["graph_snapshots"] == 22_500),
    ("26 node features", publication_summary["node_feature_dimension"] == 26),
    ("23 edge features", publication_summary["edge_feature_dimension"] == 23),
    ("13 graph-context features", publication_summary["graph_context_dimension"] == 13),
    ("15,700 train graphs", publication_summary["train_graphs"] == 15_700),
    ("3,400 validation graphs", publication_summary["validation_graphs"] == 3_400),
    ("3,400 test graphs", publication_summary["test_graphs"] == 3_400),
]

report = pd.DataFrame(checks, columns=["Check","Passed"])
report["Status"] = np.where(report["Passed"], "PASS", "FAIL")

overall = (
    "PASS_PUBLICATION_STATISTICS"
    if report["Passed"].all()
    else "REVIEW_REQUIRED"
)

report.loc[len(report)] = [
    "FINAL_PUBLICATION_STATISTICS_STATUS",
    report["Passed"].all(),
    overall
]

report.to_csv(
    OUT / "PUBLICATION_STATISTICS_REPORT.csv",
    index=False
)

display(report)

print("\nFinal status:", overall)
print("Tables:", TABLE_DIR)
print("Figures:", FIG_DIR)


## What to send back

Please upload/paste:

- `publication_statistics.json`
- `PUBLICATION_STATISTICS_REPORT.csv`
- `Table_2_graph_size_statistics.csv`
- `Table_6_graph_structural_statistics.csv`
- `Table_8_storage_footprint.csv`
- `Table_9_original_veremi_vs_veremi_graph.csv`
- `Table_11_split_graph_counts.csv`

If the status is `PASS_PUBLICATION_STATISTICS`, the next step will be:

1. create a small demo subset and example loader;
2. prepare the repository-facing dataset package;
3. start the Data in Brief manuscript using the finalized statistics.
